In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/trishi11/ntu-testing/nturgb_testing/S004C002P007R001A015_rgb.avi
/kaggle/input/datasets/trishi11/ntu-testing/nturgb_testing/S004C001P003R001A015_rgb.avi


In [2]:
!git clone https://github.com/trishaShah-web/testing123.git repo
%cd repo

Cloning into 'repo'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 68 (delta 8), reused 55 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 51.48 KiB | 6.43 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/kaggle/working/repo


In [3]:
!pip install -q torchcodec timm einops decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 94.7 MB/s eta 0:00:00


In [4]:
import torch
print("CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
!ls /kaggle/input/datasets/trishi11/ntu-testing/nturgb_testing/

CUDA: True | Tesla T4
S004C001P003R001A015_rgb.avi  S004C002P007R001A015_rgb.avi


In [5]:
!wget -q https://dl.fbaipublicfiles.com/vjepa2/vitl.pt -O /kaggle/working/vitl.pt
import torch
sd = torch.hub.load_state_dict_from_url(
    "https://dl.fbaipublicfiles.com/vjepa2/vitl.pt", map_location="cpu"
) if False else torch.load("/kaggle/working/vitl.pt", map_location="cpu")
print("checkpoint keys:", list(sd.keys())[:5])

checkpoint keys: ['encoder', 'predictor', 'opt', 'scaler', 'target_encoder']


In [6]:
!pip install -q torchcodec timm einops decord ollama

In [7]:
#!sed -i 's|http://localhost:8300|https://dl.fbaipublicfiles.com/vjepa2|g' /root/.cache/torch/hub/facebookresearch_vjepa2_main/src/hub/backbones.py

In [8]:
# Patch the poisoned URL in the cached vjepa2 checkout
!sed -i 's|http://localhost:8300|https://dl.fbaipublicfiles.com/vjepa2|g' \
  /root/.cache/torch/hub/facebookresearch_vjepa2_main/src/hub/backbones.py

# Confirm it's fixed — should now show the fbaipublicfiles URL
!grep -n "VJEPA_BASE_URL" /root/.cache/torch/hub/facebookresearch_vjepa2_main/src/hub/backbones.py

sed: can't read /root/.cache/torch/hub/facebookresearch_vjepa2_main/src/hub/backbones.py: No such file or directory
grep: /root/.cache/torch/hub/facebookresearch_vjepa2_main/src/hub/backbones.py: No such file or directory


In [9]:
import os; os.environ["PYTHONPATH"] = "."
!python scripts/smoke_test_two_clips.py \
  /kaggle/input/datasets/trishi11/ntu-testing/nturgb_testing/S004C001P003R001A015_rgb.avi \
  /kaggle/input/datasets/trishi11/ntu-testing/nturgb_testing/S004C002P007R001A015_rgb.avi \
  --blind-alpha 0.3

target:    S004C001P003R001A015_rgb.avi -> action A15 (take off jacket), performer P3
reference: S004C002P007R001A015_rgb.avi -> action A15 (take off jacket), performer P7
device: cuda
loading encoder + predictor (torch.hub, first run will download)...
Downloading: "https://github.com/facebookresearch/vjepa2/zipball/main" to /root/.cache/torch/hub/main.zip
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Downloading: "http://localhost:8300/vitl.pt" to /root/.cache/torch/hub/checkpoints/vitl.pt
Traceback (most recent call last):
  File "/usr/lib/python3.12/urllib/request.py", line 1344, in do_open
    h.request(req.get_method(), req.selector, req.data, headers,
  File "/usr/lib/python3.12/http/client.py", line 1358, in request
    self._send_request(method, url, bo